# UMA-Fold – Exploratory Analysis

This notebook is for interactive exploration of the UMA-Fold architecture and data pipeline.

Contents
1. Verify the Pairmixer block shape and parameter count
2. Visualise a pair representation
3. Inspect the data pipeline

In [ ]:
import sys
sys.path.insert(0, '..')  # allow importing from src/

import torch
from src.models.pairmixer_block import PairmixerBlock
from src.models.uma_fold import UMAFold

## 1. Block shape & parameter count

In [ ]:
pair_dim = 128
B, N = 1, 64

block = PairmixerBlock(pair_dim=pair_dim, ffn_expansion=4)
block.eval()

z = torch.randn(B, N, N, pair_dim)
out = block(z)

print(f'Input shape:  {z.shape}')
print(f'Output shape: {out.shape}')

total_params = sum(p.numel() for p in block.parameters())
print(f'Block parameters: {total_params:,}')

## 2. Full model parameter count

In [ ]:
model = UMAFold(pair_dim=128, single_dim=64, num_blocks=8)
total = sum(p.numel() for p in model.parameters())
print(f'UMAFold total parameters: {total:,}  ({total/1e6:.2f} M)')

## 3. Pair representation heatmap

In [ ]:
import matplotlib.pyplot as plt

# Run a short random sequence through the model
token_ids = torch.randint(0, 21, (1, 32))   # B=1, N=32

with torch.no_grad():
    _, pair = model.input_embedding(token_ids)
    for block in model.blocks:
        pair = block(pair)

# Mean over channel dimension → [N, N]
pair_map = pair[0].float().mean(dim=-1).detach().numpy()

plt.figure(figsize=(6, 5))
plt.imshow(pair_map, cmap='RdBu_r', aspect='auto')
plt.colorbar(label='mean activation')
plt.title('Pair representation (mean over channels)')
plt.xlabel('Residue j')
plt.ylabel('Residue i')
plt.tight_layout()
plt.show()